# Etapa 2 — Screening de Selección (`selection_campaign_1200_50`) y Poda de Modelos

Replica `tools/selection_campaign_1200_50.py` e implementa la lógica de **screening cuantitativo en dos pasadas** con **detección temprana de divergencia**:
1. **Pasada 1 (`with_energy`, `"energy": 100.0`)** y **Pasada 2 (`no_energy`, `"energy": 0.0`)**: entrena todas las arquitecturas candidatas con presupuesto intermedio (`4000 Adam + 200 L-BFGS` en `full_campaign`; escala rápida en `demo_fast`) sobre los casos pivote (`mms_sine`, `cable_disk`, `xlpe_discrete_layers`, `aras_flat` en `full_campaign`; `mms_sine`, `cable_disk`, `xlpe_single` en `demo_fast`), evaluando frente a referencia analítica y **FEM WSL2**. Si una corrida diverge (`NaN`/`Inf`, explosión o crecimiento sostenido de pérdida), `divergence_stop` la detiene inmediatamente sin esperar a completar todos los pasos.
2. Calcula el puntaje compuesto adimensional (error en $T_{\max}$, `NRMSE`, desbalance energético y robustez en cables multicapa/planos frente a `mms_sine`) sobre **ambas pasadas** para seleccionar el mejor modo energético de cada arquitectura y filtrar modelos que se estancan o divergen.
3. Escribe la lista curada de **modelos promovidos** en `campaign_workspace/promoted_models.json`, reteniendo explícitamente si cada modelo clasificó **con `energy`** (`100.0`) o **sin `energy`** (`0.0`) en la función de pérdida.

In [11]:
import json
import os
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "full_campaign"    #"demo_fast"

# Modo de sobreescritura / reutilización desde disco:
# - OVERWRITE = False -> carga resultados ya guardados en disco y solo ejecuta corridas faltantes
# - OVERWRITE = True  -> ignora caché previa y vuelve a entrenar/calcular todo desde cero
OVERWRITE = False

# Paralelismo de campaña:
# - USE_PARALLEL=True  -> fuerza device='cpu' con ParallelRunner multi-proceso
# - USE_PARALLEL=False -> ejecuta secuencialmente en device='auto' (p. ej. XPU nativo sin fallbacks)
USE_PARALLEL = True
MAX_WORKERS = min(12, os.cpu_count() or 1)
DEVICE = "cpu" if USE_PARALLEL else "auto"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | OVERWRITE={OVERWRITE} | USE_PARALLEL={USE_PARALLEL} (workers={MAX_WORKERS}, device={DEVICE!r}) | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='full_campaign' | OVERWRITE=False | USE_PARALLEL=True (workers=12, device='cpu') | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [12]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        eval_cache = fem_dir / "eval_ref_T.npz"
        if eval_cache.is_file() and eval_cache.stat().st_mtime >= fem_json.stat().st_mtime:
            try:
                cached_T = np.asarray(np.load(eval_cache)["T"], dtype=float)
                if cached_T.shape == (len(xy),) and np.isfinite(cached_T).all():
                    return cached_T, f"FEM_WSL_cache:{case_id}"
            except Exception:
                pass
        art = SolutionArtifact.load(fem_json)
        ref_T = np.asarray(art.predict(xy)["T"], dtype=float)
        try:
            np.savez_compressed(eval_cache, T=ref_T)
        except OSError:
            pass
        return ref_T, f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    ifaces = ev.get("interfaces") or []
    worst_flux_jump = max((item.get("derived_flux_jump_rms_pct", 0.0) for item in ifaces), default=None)
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "interface_flux_jump_rms_pct": worst_flux_jump,
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "best_loss": result.get("best_loss"),
        "elapsed_s": result.get("elapsed_s"),
    }
CASES_STAGE2 = (
    ("mms_sine", "cable_disk", "xlpe_single")
    if RUN_MODE == "demo_fast"
    else ("mms_sine", "cable_disk", "xlpe_discrete_layers", "aras_flat")
)
CANDIDATE_MODELS_S2 = (
    ("M0", "M1", "M1_EXACT", "CUSPR", "M0R", "M2")
    if RUN_MODE == "demo_fast"
    else ("M0", "M1", "M2", "DCSNN", "M0R", "CUSP", "CUSPR", "FBPINN", "M1_EXACT", "CUSP_EXACT")
)
ADAM_S2 = 400 if RUN_MODE == "demo_fast" else 4000
LBFGS_S2 = 15 if RUN_MODE == "demo_fast" else 200

ENERGY_PASSES_S2 = (
    {"label": "with_energy", "use_energy_loss": True, "energy_weight": 100.0},
    {"label": "no_energy", "use_energy_loss": False, "energy_weight": 0.0},
)
DIVERGENCE_STOP_S2 = {
    "max_loss": 1e12,
    "explosion_factor": 1e4,
    "patience_explosion": 3,
    "growth_patience": 6,
    "min_growth_ratio": 1.05,
    "warmup_steps": 150,
    "lbfgs_explosion_factor": 100.0,
    "lbfgs_stagnation_checks": 8,
    "check_nonfinite_grads": True,
}

stage2_dir = CAMP_WS / "stage2_selection_4000_200"
stage2_dir.mkdir(parents=True, exist_ok=True)
stage2_summary_path = CAMP_WS / "stage2_selection_summary.json"

def run_stage2_pass(pass_cfg: dict, overwrite: bool | None = None) -> dict:
    """Ejecuta una pasada completa de screening (con o sin término energy) o reutiliza resultados en disco."""
    do_overwrite = OVERWRITE if overwrite is None else bool(overwrite)
    label = str(pass_cfg["label"])
    use_energy = bool(pass_cfg["use_energy_loss"])
    energy_weight = float(pass_cfg["energy_weight"])
    raw_summary = json.loads(stage2_summary_path.read_text(encoding="utf-8-sig")) if stage2_summary_path.is_file() else {}
    # Elimina llaves legacy sin etiqueta de pasada (case:arch) para evitar colisiones
    summary = {k: v for k, v in raw_summary.items() if k.count(":") >= 2}
    tasks = []
    reused_count = 0
    dirty = False
    case_cache = {}
    for case_id in CASES_STAGE2:
        for arch in CANDIDATE_MODELS_S2:
            key = f"{case_id}:{arch}:{label}"
            run_dir = stage2_dir / label / case_id / arch
            if not do_overwrite:
                cached = summary.get(key)
                if cached is not None and cached.get("status") in ("completed", "diverged"):
                    reused_count += 1
                    if cached["status"] == "diverged":
                        div = cached.get("divergence_stop") or {}
                        print(f"[Etapa 2 | {label} | CACHED] {key:42s} | DIVERGED @ step={div.get('step')} ({div.get('reason')})")
                    else:
                        print(f"[Etapa 2 | {label} | CACHED] {key:42s} | NRMSE={cached.get('nrmse_pct')} | dTmax={cached.get('error_Tmax_abs_K')} K | bal={cached.get('balance_pct')}%")
                    continue
                res_file = run_dir / "result.json"
                if res_file.is_file():
                    res_disk = json.loads(res_file.read_text(encoding="utf-8-sig"))
                    if res_disk.get("status") in ("completed", "diverged"):
                        m = extract_metrics(res_disk)
                        rec = {
                            "case": case_id,
                            "architecture": arch,
                            "energy_mode": label,
                            "use_energy_loss": use_energy,
                            "energy_weight": energy_weight,
                            "exact_energy": use_energy,
                            "status": res_disk.get("status"),
                            "divergence_stop": res_disk.get("divergence_stop"),
                            **m,
                        }
                        if res_disk.get("error"):
                            rec["error"] = res_disk.get("error")
                        summary[key] = rec
                        dirty = True
                        reused_count += 1
                        if rec["status"] == "diverged":
                            div = rec.get("divergence_stop") or {}
                            print(f"[Etapa 2 | {label} | CACHED result.json] {key:42s} | DIVERGED @ step={div.get('step')} ({div.get('reason')})")
                        else:
                            print(f"[Etapa 2 | {label} | CACHED result.json] {key:42s} | dev={res_disk.get('device')} | NRMSE={m['nrmse_pct']} | dTmax={m['error_Tmax_abs_K']} K | bal={m['balance_pct']}%")
                        continue
            if case_id not in case_cache:
                prob = catalog.load(case_id)
                ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
                case_cache[case_id] = (prob, ref_vals, ref_lbl)
            prob, ref_vals, ref_lbl = case_cache[case_id]
            tasks.append(
                RunTask(
                    name=key,
                    problem_source=prob.doc,
                    architecture=arch,
                    output_dir=run_dir,
                    config_kwargs={
                        "adam_steps": ADAM_S2,
                        "lbfgs_steps": LBFGS_S2,
                        "lr": 2e-3,
                        "width": 32,
                        "depth": 3,
                        "weights": {"pde": 1.0, "energy": energy_weight},
                        "exact_energy": use_energy,
                        "sampling": "mixed",
                        "focused": True,
                        "focus_criterion": "residual",
                        "focus_surfaces": False,
                        "resample_every": 100,
                        "full_audit": True,
                        "divergence_stop": DIVERGENCE_STOP_S2,
                        "device": DEVICE,
                        "dtype": "float32",
                    },
                    reference=ref_vals,
                    reference_label=ref_lbl,
                    overwrite=True,
                )
            )

    def _on_done(res: dict) -> None:
        key = res["name"]
        case_id, arch, _ = key.split(":", 2)
        m = extract_metrics(res)
        rec = {
            "case": case_id,
            "architecture": arch,
            "energy_mode": label,
            "use_energy_loss": use_energy,
            "energy_weight": energy_weight,
            "exact_energy": use_energy,
            "status": res.get("status"),
            "divergence_stop": res.get("divergence_stop"),
            **m,
        }
        if res.get("error"):
            rec["error"] = res.get("error")
        summary[key] = rec
        write_json_atomic(stage2_summary_path, summary)
        if rec["status"] == "diverged":
            div = rec.get("divergence_stop") or {}
            print(f"[Etapa 2 | {label}] {key:42s} | DIVERGED @ step={div.get('step')} ({div.get('reason')})")
        elif rec["status"] != "completed":
            print(f"[Etapa 2 | {label}] {key:42s} | FAILED: {rec.get('error')}")
        else:
            print(f"[Etapa 2 | {label}] {key:42s} | dev={res.get('device')} | NRMSE={m['nrmse_pct']} | dTmax={m['error_Tmax_abs_K']} K | bal={m['balance_pct']}%")

    if tasks:
        runner = ParallelRunner(max_workers=MAX_WORKERS if USE_PARALLEL else 1, distribute_devices=not USE_PARALLEL)
        runner.run(tasks, on_complete=_on_done, show_progress=True, progress_title=f"Etapa 2 — {label} (energy={energy_weight:g})")
        write_json_atomic(stage2_summary_path, summary)
    else:
        if dirty:
            write_json_atomic(stage2_summary_path, summary)
        print(f"[Etapa 2 | {label}] Todas las {reused_count} corridas ya estaban guardadas en disco (OVERWRITE=False); no se re-entrenó nada.")
    return summary

In [13]:
# Pasada 1: Screening con término energético ("energy": 100.0)
stage2_summary = run_stage2_pass(ENERGY_PASSES_S2[0])

[Etapa 2 | with_energy | CACHED] mms_sine:M0:with_energy                    | NRMSE=0.2841128687534951 | dTmax=0.16809263017277942 K | bal=0.002346645468369784%
[Etapa 2 | with_energy | CACHED] mms_sine:M1:with_energy                    | NRMSE=0.34628832629886164 | dTmax=0.06589307572942005 K | bal=0.002170487161705234%
[Etapa 2 | with_energy | CACHED] mms_sine:M2:with_energy                    | NRMSE=0.7065992121063871 | dTmax=0.1553477266083263 K | bal=0.03284337844660133%
[Etapa 2 | with_energy | CACHED] mms_sine:DCSNN:with_energy                 | NRMSE=0.36536233365563897 | dTmax=0.10080137040715442 K | bal=7.470084329203175e-06%
[Etapa 2 | with_energy | CACHED] mms_sine:M0R:with_energy                   | NRMSE=0.3007049783911972 | dTmax=0.12412061479192005 K | bal=0.0016996640511654365%
[Etapa 2 | with_energy | CACHED] mms_sine:CUSP:with_energy                  | NRMSE=0.34628832629886164 | dTmax=0.06589307572942005 K | bal=0.002170487161705234%
[Etapa 2 | with_energy | CACHED

## Pasada 2 — Screening sin término energético (`"energy": 0.0`)

In [14]:
# Pasada 2: Screening sin término energético ("energy": 0.0)
stage2_summary = run_stage2_pass(ENERGY_PASSES_S2[1])

[Etapa 2 | no_energy | CACHED] mms_sine:M0:no_energy                      | NRMSE=0.019560151481415176 | dTmax=0.008779623245189327 K | bal=0.7702671459545368%
[Etapa 2 | no_energy | CACHED] mms_sine:M1:no_energy                      | NRMSE=0.2816609443679973 | dTmax=0.1423207304229237 K | bal=0.6624214291753479%
[Etapa 2 | no_energy | CACHED] mms_sine:M2:no_energy                      | NRMSE=0.803133196335621 | dTmax=0.6666584989776112 K | bal=0.44272319485992695%
[Etapa 2 | no_energy | CACHED] mms_sine:DCSNN:no_energy                   | NRMSE=0.41943762041220894 | dTmax=0.3223782560576893 K | bal=1.5213741367891482%
[Etapa 2 | no_energy | CACHED] mms_sine:M0R:no_energy                     | NRMSE=0.18098066095143123 | dTmax=0.025365731979420048 K | bal=0.1036232588481226%
[Etapa 2 | no_energy | CACHED] mms_sine:CUSP:no_energy                    | NRMSE=0.2816609443679973 | dTmax=0.1423207304229237 K | bal=0.6624214291753479%
[Etapa 2 | no_energy | CACHED] mms_sine:CUSPR:no_energy 

## Lógica de Screening y Promoción de Modelos

In [15]:
# Agregación multi-caso y multi-pasada (with_energy vs no_energy) + regla de poda acotada
if stage2_summary_path.is_file():
    stage2_summary = json.loads(stage2_summary_path.read_text(encoding="utf-8-sig"))

def _resolve_loss(rec: dict):
    div = rec.get("divergence_stop") or {}
    for cand in (rec.get("best_loss"), (div.get("lbfgs_early_stop") or {}).get("loss"), div.get("best_loss")):
        if cand is not None:
            try:
                val = float(cand)
                if np.isfinite(val):
                    return val
            except (TypeError, ValueError):
                pass
    return None

# Deduplica registros por (architecture, energy_mode, case): prioriza llaves explícitas (case:arch:mode) sobre legacy
dedup_records = {}
for key, rec in stage2_summary.items():
    case_id = rec.get("case")
    arch = rec.get("architecture")
    if case_id not in CASES_STAGE2 or not arch:
        continue
    is_explicit = key.count(":") >= 2
    energy_mode = rec.get("energy_mode") or (key.split(":", 2)[2] if is_explicit else "with_energy")
    rkey = (arch, energy_mode, case_id)
    prev = dedup_records.get(rkey)
    if prev is None or (is_explicit and not prev[0]):
        dedup_records[rkey] = (is_explicit, key, rec)

per_variant = {}
diverged_runs = []
for (arch, energy_mode, case_id), (_is_explicit, key, rec) in sorted(dedup_records.items()):
    use_energy = bool(rec["use_energy_loss"]) if "use_energy_loss" in rec else (energy_mode == "with_energy")
    energy_weight = float(rec["energy_weight"]) if "energy_weight" in rec else (100.0 if use_energy else 0.0)
    exact_energy = bool(rec["exact_energy"]) if "exact_energy" in rec else use_energy

    if rec.get("status") == "diverged":
        div = rec.get("divergence_stop") or {}
        diverged_runs.append({
            "key": key,
            "case": case_id,
            "architecture": arch,
            "energy_mode": energy_mode,
            "energy_weight": energy_weight,
            "step": div.get("step"),
            "phase": div.get("phase"),
            "reason": div.get("reason"),
        })
        continue
    if rec.get("status") != "completed":
        continue

    vkey = (arch, energy_mode)
    row = per_variant.setdefault(
        vkey,
        {
            "architecture": arch,
            "energy_mode": energy_mode,
            "use_energy_loss": use_energy,
            "energy_weight": energy_weight,
            "exact_energy": exact_energy,
            "log_scores": [],
            "linear_scores": [],
            "nrmse": [],
            "balance": [],
            "dtmax": [],
            "flux_jump": [],
            "sanity_failures": [],
            "strict_passed": 0,
            "elapsed": [],
        },
    )
    bal = rec["balance_pct"] if rec.get("balance_pct") is not None else 100.0
    nrmse = rec["nrmse_pct"] if rec.get("nrmse_pct") is not None else 100.0
    dtmax = rec["error_Tmax_abs_K"] if rec.get("error_Tmax_abs_K") is not None else 100.0
    linear_score = (dtmax / STRICT_CRITERIA["error_Tmax_abs_K"]) + (nrmse / STRICT_CRITERIA["nrmse_pct"]) + (bal / STRICT_CRITERIA["balance_pct"])
    log_score = float(
        np.log1p(dtmax / STRICT_CRITERIA["error_Tmax_abs_K"])
        + np.log1p(nrmse / STRICT_CRITERIA["nrmse_pct"])
        + np.log1p(bal / STRICT_CRITERIA["balance_pct"])
    )
    loss_val = _resolve_loss(rec)
    if loss_val is not None and not np.isfinite(loss_val):
        row["sanity_failures"].append(f"{case_id}:nonfinite_loss")
    if case_id == "mms_sine":
        if nrmse >= 5.0:
            row["sanity_failures"].append(f"mms_sine:nrmse={nrmse:.2f}%>=5%")
        if bal >= 10.0:
            row["sanity_failures"].append(f"mms_sine:bal={bal:.2f}%>=10%")
        if dtmax >= 2.0:
            row["sanity_failures"].append(f"mms_sine:dTmax={dtmax:.2f}K>=2K")
        if loss_val is not None and loss_val >= 10.0:
            row["sanity_failures"].append(f"mms_sine:loss={loss_val:.2f}>=10")
    elif bal >= 50.0:
        row["sanity_failures"].append(f"{case_id}:bal={bal:.2f}%>=50%")

    row["log_scores"].append(log_score)
    row["linear_scores"].append(linear_score)
    row["nrmse"].append(nrmse)
    row["balance"].append(bal)
    row["dtmax"].append(dtmax)
    if rec.get("interface_flux_jump_rms_pct") is not None:
        row["flux_jump"].append(float(rec["interface_flux_jump_rms_pct"]))
    if dtmax < STRICT_CRITERIA["error_Tmax_abs_K"] and nrmse < STRICT_CRITERIA["nrmse_pct"] and bal < STRICT_CRITERIA["balance_pct"]:
        row["strict_passed"] += 1
    row["elapsed"].append(rec.get("elapsed_s") or 0.0)

def _rank_key(r: dict) -> tuple:
    return (
        not r["sanity_passed"],
        r["mean_score"],
        r["max_balance_pct"],
        r["mean_nrmse_pct"],
        -r["strict_cases_passed"],
    )

ranking = []
for (_arch, _mode), vals in per_variant.items():
    if len(vals["log_scores"]) == len(CASES_STAGE2):
        ranking.append({
            "architecture": vals["architecture"],
            "energy_mode": vals["energy_mode"],
            "use_energy_loss": vals["use_energy_loss"],
            "energy_weight": vals["energy_weight"],
            "exact_energy": vals["exact_energy"],
            "sanity_passed": len(vals["sanity_failures"]) == 0,
            "sanity_failures": list(vals["sanity_failures"]),
            "mean_score": float(np.mean(vals["log_scores"])),
            "mean_log_score": float(np.mean(vals["log_scores"])),
            "mean_linear_score": float(np.mean(vals["linear_scores"])),
            "max_balance_pct": float(np.max(vals["balance"])),
            "mean_nrmse_pct": float(np.mean(vals["nrmse"])),
            "mean_error_Tmax_abs_K": float(np.mean(vals["dtmax"])),
            "max_flux_jump_rms_pct": float(np.max(vals["flux_jump"])) if vals["flux_jump"] else None,
            "strict_cases_passed": int(vals["strict_passed"]),
            "total_elapsed_s": float(np.sum(vals["elapsed"])),
        })
ranking.sort(key=_rank_key)

# Mejor configuración (con energy o sin energy) por arquitectura evaluando ambas pasadas
THESIS_RETAINED = ["M1", "M1_EXACT", "CUSPR", "M0R", "M2"]
by_arch = {}
for item in ranking:
    by_arch.setdefault(item["architecture"], []).append(item)

best_per_architecture = []
for arch, variants in by_arch.items():
    variants_sorted = sorted(variants, key=_rank_key)
    winner = dict(variants_sorted[0])
    winner["in_thesis_retained"] = arch in THESIS_RETAINED
    if len(variants_sorted) > 1:
        other = variants_sorted[1]
        winner["comparison_other_pass"] = {
            "energy_mode": other["energy_mode"],
            "energy_weight": other["energy_weight"],
            "sanity_passed": other["sanity_passed"],
            "mean_score": other["mean_score"],
            "mean_nrmse_pct": other["mean_nrmse_pct"],
            "max_balance_pct": other["max_balance_pct"],
            "strict_cases_passed": other["strict_cases_passed"],
        }
    else:
        other_divs = [d for d in diverged_runs if d["architecture"] == arch and d["energy_mode"] != winner["energy_mode"]]
        winner["comparison_other_pass"] = {"status": "diverged", "runs": other_divs} if other_divs else None
    best_per_architecture.append(winner)
best_per_architecture.sort(key=_rank_key)

sane_candidates = [r for r in best_per_architecture if r["sanity_passed"]]
rejected_sanity = [r for r in best_per_architecture if not r["sanity_passed"]]
promoted_models_detail = (sane_candidates if sane_candidates else best_per_architecture)[:5]
promoted_models = [r["architecture"] for r in promoted_models_detail]
long_run_finalists_detail = promoted_models_detail[:2]
finalists = [r["architecture"] for r in long_run_finalists_detail]
promoted_energy_config = {
    r["architecture"]: {
        "energy_mode": r["energy_mode"],
        "use_energy_loss": r["use_energy_loss"],
        "energy_weight": r["energy_weight"],
        "exact_energy": r["exact_energy"],
    }
    for r in promoted_models_detail
}

payload = {
    "run_mode": RUN_MODE,
    "ranking_criterion": "log1p_composite_with_sanity_gate",
    "energy_passes": list(ENERGY_PASSES_S2),
    "divergence_stop": DIVERGENCE_STOP_S2,
    "ranking": ranking,
    "best_per_architecture": best_per_architecture,
    "rejected_sanity": rejected_sanity,
    "diverged_runs": diverged_runs,
    "promoted_models": promoted_models,
    "promoted_models_detail": promoted_models_detail,
    "promoted_energy_config": promoted_energy_config,
    "long_run_finalists": finalists,
    "long_run_finalists_detail": long_run_finalists_detail,
}
write_json_atomic(CAMP_WS / "promoted_models.json", payload)

print("=== Modelos promovidos (considerando ambas pasadas: con/sin energy) ===")
for r in promoted_models_detail:
    tag = "CON energy (100.0)" if r["use_energy_loss"] else "SIN energy (0.0)"
    print(
        f"- {r['architecture']:10s} | clasificó: {tag:18s} | log_score={r['mean_score']:6.3f} "
        f"| strict={r['strict_cases_passed']}/{len(CASES_STAGE2)} | NRMSE={r['mean_nrmse_pct']:.4f}% "
        f"| max_bal={r['max_balance_pct']:.4f}% | thesis={r['in_thesis_retained']}"
    )
if rejected_sanity:
    print("\nModelos descartados por compuerta de sanidad:")
    for r in rejected_sanity:
        print(f"- {r['architecture']:10s} ({r['energy_mode']}): {', '.join(r['sanity_failures'])}")
if diverged_runs:
    print(f"\nCorridas detenidas por divergencia ({len(diverged_runs)}):", json.dumps(diverged_runs, indent=2))
print("\nModelos promovidos a Etapa 3/4:", promoted_models)
print("Finalistas corridas largas:", finalists)

=== Modelos promovidos (considerando ambas pasadas: con/sin energy) ===
- M0R        | clasificó: CON energy (100.0) | log_score= 5.206 | strict=0/4 | NRMSE=0.9914% | max_bal=3.4931% | thesis=True
- M1         | clasificó: CON energy (100.0) | log_score= 5.253 | strict=0/4 | NRMSE=0.8404% | max_bal=4.4899% | thesis=True
- FBPINN     | clasificó: CON energy (100.0) | log_score= 5.279 | strict=0/4 | NRMSE=0.6390% | max_bal=6.5706% | thesis=False
- M2         | clasificó: CON energy (100.0) | log_score= 7.389 | strict=0/4 | NRMSE=5.3241% | max_bal=5.1057% | thesis=True
- DCSNN      | clasificó: CON energy (100.0) | log_score= 7.827 | strict=0/4 | NRMSE=5.3543% | max_bal=7.7494% | thesis=False

Modelos descartados por compuerta de sanidad:
- CUSPR      (with_energy): cable_disk:bal=584.23%>=50%
- CUSP_EXACT (no_energy): aras_flat:bal=2161.49%>=50%, cable_disk:bal=100.49%>=50%
- M0         (with_energy): aras_flat:bal=84.47%>=50%, xlpe_discrete_layers:bal=57.19%>=50%
- CUSP       (with_ener